# Beadform: free schema benchmark
Use an eligible **free** Colab GPU. This notebook does not need Google Drive, paid compute, or API keys. It evaluates twelve original prompts; it does not train on them. Results measure JSON schema structure, not visual accuracy or physical buildability.
Choose Runtime → Change runtime type → a GPU offered on the free tier. If none is available, stop and retry later. Use one account and honor Colab limits.


In [ ]:
from pathlib import Path
import subprocess, sys, torch
if not torch.cuda.is_available():
    raise RuntimeError("No free CUDA GPU offered. Retry later; do not upgrade.")
print("GPU:", torch.cuda.get_device_name(0))
repo = Path("/content/beadform-schema")
if not repo.exists():
    subprocess.run(["git","clone","--depth","1","--branch","runpod-training-guide","https://github.com/c-tustin/beadform.git",str(repo)],check=True)
print("Code revision:",subprocess.check_output(["git","rev-parse","HEAD"],cwd=repo,text=True).strip())
subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(repo/"training/requirements.txt")],check=True)


In [ ]:
import json
out = Path("/content/beadform-schema-results")
out.mkdir(exist_ok=True)
summary = out/"summary.json"
if not summary.exists():
    command = [sys.executable,"-u",str(repo/"training/evaluate_lora.py"),
        "--data",str(repo/"training/benchmarks/schema-fresh-12.jsonl"),
        "--output",str(out/"predictions.jsonl"),"--summary",str(summary),
        "--bounded","--prompts-only","--resume"]
    with (out/"run.log").open("a") as log:
        result = subprocess.run(command,stdout=log,stderr=subprocess.STDOUT)
    print((out/"run.log").read_text()[-6000:])
    result.check_returncode()
print(summary.read_text())
print("Review every scene separately for subject accuracy and construction quality.")


In [ ]:
# Download results before the temporary free runtime ends.
import shutil
from google.colab import files
archive = shutil.make_archive("/content/beadform-schema-results","zip",out)
files.download(archive)
